# 04: TF-IDF + cosine blocking against the FULL S2/S3 pools
Honest recall: the pools are the complete train sources (normalized by `work/norm_full.py`), not the small dev sample. Memory-safe: per (source, country), small query chunks, absolute document-frequency cap.

In [1]:
import os, sys, time, gc
os.environ['TMP']=os.environ['TEMP']='D:/tmp'
sys.path.insert(0,'D:/Amazon_ML_Challenge')
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from src.norm import norm_name, norm_addr
W='D:/Amazon_ML_Challenge/work/'; TR='D:/Amazon_ML_Challenge/Hackathon/Datasets/Train Datasets/'
NQ=5000
gt=pd.read_csv(TR+'train_ground_truth.tsv',sep='\t',dtype=str,keep_default_na=False,quoting=3)
gt=gt.sample(NQ,random_state=0)
need=set(gt.source1_entity_id)
q=pd.concat([c[c.entity_id.isin(need)] for c in pd.read_csv(TR+'train_source1.tsv',sep='\t',dtype=str,keep_default_na=False,quoting=3,chunksize=500000)]).reset_index(drop=True)
q['nn']=q.business_name.map(norm_name); q['na']=[norm_addr(a,c) for a,c in zip(q.business_address,q.country)]
g=gt.set_index('source1_entity_id').matched_entity_ids.str.split(',')
print(len(q), q.country.value_counts().to_dict())

5000 {'US': 2961, 'India': 2039}


In [2]:
def topk_cosine(Q,T,k,chunk=50):
    Tt=T.T.tocsr(); out=[]
    for i in range(0,Q.shape[0],chunk):
        S=(Q[i:i+chunk]@Tt).tocsr()
        for r in range(S.shape[0]):
            a,b=S.indptr[r],S.indptr[r+1]; ix=S.indices[a:b]; v=S.data[a:b]
            if len(v)>k: p=np.argpartition(-v,k)[:k]; ix,v=ix[p],v[p]
            out.append((ix,v))
    return out

K=20; MAXDF=20000
def run(src):
    """returns list (per query) of dict channel -> (entity_ids, scores)"""
    res=[{} for _ in range(len(q))]
    for c in q.country.unique():
        qi=np.flatnonzero(q.country.values==c)
        pool=pd.read_parquet(W+f'full/{src}_{c}.parquet'); eid=pool.entity_id.values
        for ch,col in (('name','nn'),('addr','na')):
            vec=TfidfVectorizer(token_pattern=r"\S+",lowercase=False,sublinear_tf=True,max_df=MAXDF,dtype=np.float32)
            T=vec.fit_transform(pool[col].values); Q=vec.transform(q[col].values[qi])
            for j,(ix,v) in zip(qi,topk_cosine(Q,T,K)): res[j][ch]=(eid[ix],v)
            del T,Q,vec; gc.collect()
        print(src,c,len(pool),'rows done',round(time.time()-t0),'s',flush=True); del pool; gc.collect()
    return res
t0=time.time(); R={s:run(s) for s in ('s2','s3')}

s2 India 2017799 rows done 100 s


s2 US 3016817 rows done 240 s


s3 India 2115547 rows done 333 s


s3 US 3170056 rows done 515 s


## Recall on the full pools

In [3]:
def evaluate(src,by_country=True):
    rows=[]
    for i,qid in enumerate(q.entity_id):
        truth={t for t in g[qid] if t.startswith(src.upper())}
        if not truth: continue
        n=set(R[src][i]['name'][0]); a=set(R[src][i]['addr'][0])
        rows.append((q.country[i],len(truth),len(truth&n),len(truth&a),len(truth&(n|a)),len(n|a)))
    d=pd.DataFrame(rows,columns=['country','truth','name','addr','union','ncand'])
    tot=d.groupby('country').agg(truth=('truth','sum'),name=('name','sum'),addr=('addr','sum'),union=('union','sum'),avg_cands=('ncand','mean'))
    for c in ('name','addr','union'): tot[c]=(tot[c]/tot.truth).round(3)
    all_=d[['truth','name','addr','union']].sum(); print(src,'ALL',{c:round(all_[c]/all_.truth,3) for c in ('name','addr','union')},'avg cands',round(d.ncand.mean(),1))
    return tot
for s in ('s2','s3'): display(evaluate(s))

s2 ALL {'name': np.float64(0.594), 'addr': np.float64(0.868), 'union': np.float64(0.948)} avg cands 38.5


,truth,name,addr,union,avg_cands
country,,,,,
India,3392,0.510,0.887,0.937,38.622524
US,4941,0.652,0.854,0.956,38.445703


s3 ALL {'name': np.float64(0.579), 'addr': np.float64(0.84), 'union': np.float64(0.932)} avg cands 38.5


,truth,name,addr,union,avg_cands
country,,,,,
India,3589,0.502,0.786,0.898,38.687252
US,5147,0.632,0.877,0.955,38.378096


In [4]:
# save candidates with cosine scores for the matcher stage
rows=[]
for s in ('s2','s3'):
    for i,qid in enumerate(q.entity_id):
        for ch,(e,v) in R[s][i].items(): rows+= [(qid,s,ch,x,float(y)) for x,y in zip(e,v)]
cand=pd.DataFrame(rows,columns=['s1_id','src','channel','cand_id','cos'])
cand.to_parquet(W+'cand_full_sample.parquet'); print(len(cand))

396974
